# Module 43 — Object Segmentation

## 4. Mask Representation

**Must Know — PyTorch + Computer Vision**

---

Now that we understand semantic and instance segmentation, let's understand how segmentation masks are actually represented and stored.

A segmentation mask is a numerical representation that tells us which pixels belong to which class or object.

Unlike an ordinary RGB image, a segmentation mask usually stores labels rather than natural image colours.

There are three important representations:

1. Binary masks
2. Multi-class masks
3. Instance IDs

### 4.1 Binary Masks

A binary mask contains only two possible values:

- `0` → Background
- `1` → Foreground (object)

Suppose we have an image containing one dog.

![Example of a binary segmentation mask showing foreground objects in white and background in black](https://bioimagingai.janelia.org/9-figures/chapter9_1_segmentation_types.png)

*Source: [AI in Microscopy — Segmentation Types, Janelia Research Campus](https://bioimagingai.janelia.org/9-train-models.html)*

**Original image references:**

- [Happy dog on white background](https://images.openai.com/static-rsc-4/F8k8luUZkk93RW3IEXeWM1sYqPJAA7hjj9jaVfK_mLoDD0wR4V9ArnDAgisEEg39gjRPE7ZasM50Y8vgcBlH60_UJDmh3KQnhZtPDWSnTmfMbStUCiDC9Dlr2WDS6ePbXynCCK8vjM_YBzO_z9uODuJiWeg4e4NUFaS97qpsFxpgdyl6QVyI5gdV_5qT_-qK?purpose=inline)
- [Segment Anything — GitHub Issue #592](https://images.openai.com/static-rsc-4/wpN8HGmN9SaXB88mWuLsJsmmE29vPvhZJ7Q8-usi-TPo76EtSAFu4hjYQPe92cLcXjX6LWfOQA4FQ0-AJ_Jpq5I27PlF6OPRxp6TiubmUVj3yl9k-4t1JcgVI_fhWaL7X7Lgfcrl-ZBLxfEAWx-ITFKQDOfOODuaIL6QTmg72bo?purpose=inline)

The corresponding binary mask could look like this:

```text
0 0 0 0 0 0
0 1 1 1 0 0
0 1 1 1 0 0
0 1 1 1 1 0
0 0 1 1 0 0
0 0 0 0 0 0
```

Here:

- `1` represents pixels belonging to the dog.
- `0` represents background pixels.

Mathematically:

$$
M \in \{0,1\}^{H \times W}
$$

Where:

- \\(M\\) = Binary mask.
- \\(H\\) = Image height.
- \\(W\\) = Image width.

Practical applications:

- Background removal.
- Tumor region detection.
- Defect detection.
- Foreground extraction.

**Important:** Binary masks are particularly common in binary segmentation tasks, where we distinguish one foreground category from the background.

### 4.2 Multi-Class Masks

Now imagine an autonomous vehicle camera image.

It contains:

- Road
- Cars
- Pedestrians
- Buildings
- Trees
- Sky

A binary mask cannot represent all these classes independently in a single label map.

We therefore use a multi-class mask.

![Semantic segmentation visualization showing different classes represented by different colors](https://bioimagingai.janelia.org/9-figures/chapter9_1_segmentation_types.png)

*Source: [AI in Microscopy — Segmentation Types, Janelia Research Campus](https://bioimagingai.janelia.org/9-train-models.html)*

**Original reference:**

- [Planning Shaded Corridors to Mitigate Heat — MDPI](https://images.openai.com/static-rsc-4/rACWOBAYAB5jVrjfJm_sc9B7UwIazS8bylz6H1-3IzCKUPf4qNjIAtJtYK4IzNG26lJDOaJ5V8-nHjFKdsm_AVHc4_rxfpAsLnufNZvXjOp61n9X4fKLFUX3q8OU-e5qROhXV2uU8wswbCSCcoFKxfFA_UXuL_BfN86UyXmKSWY?purpose=inline)
- [mdpi.com](https://www.mdpi.com/2073-445X/15/5/739?utm_source=chatgpt.com)

Each pixel receives an integer representing its class.

Example:

```python
classes = {
    0: "Background",
    1: "Road",
    2: "Car",
    3: "Person",
    4: "Building",
    5: "Tree"
}
```

A simplified mask:

```text
0 0 4 4 4 5
0 0 4 4 5 5
1 1 1 1 1 1
1 1 2 2 1 1
1 1 3 1 1 1
```

Each integer represents a different semantic class.

The mathematical representation is:

$$
M \in \{0,1,\ldots,C-1\}^{H \times W}
$$

Where:

- \\(C\\) = Number of classes.
- \\(H,W\\) = Mask dimensions.

For example, if we have six classes:

$$
M \in \{0,1,2,3,4,5\}^{256 \times 256}
$$

The mask stores class IDs, not RGB colours.

**Important PyTorch connection:**

For multi-class semantic segmentation, the typical target mask has shape:

```python
[B, H, W]
```

And generally uses integer class IDs (`torch.long`) with `CrossEntropyLoss`.

### 4.3 Instance IDs

Now let's move to instance segmentation.

Suppose an image contains three cars.

All three cars belong to the same semantic category: `Car`.

But we want to identify each car independently.

This is where Instance IDs become important.

![Instance segmentation showing individual objects with different colors](https://bioimagingai.janelia.org/9-figures/chapter9_1_segmentation_types.png)

*Source: [AI in Microscopy — Segmentation Types, Janelia Research Campus](https://bioimagingai.janelia.org/9-train-models.html)*

**Original reference:**

- [Revolutionizing Autonomous Driving with Instance Segmentation — Keylabs](https://images.openai.com/static-rsc-4/S7YhnDspdcB-ykbbm0aFanx0T_ACwmrvdACspAghr7LHzYQT_3EkAS9kX6do93Et5TKg4BUtiME41d6NBlnxHvzc9eR9wAWSyCUIRILE7uzik6BpXTY4Gipb1dO3UT1fqfS9TzGeJ0EQSxhLS4gTkbWDgYjZJZ28shwQ0ge328Q?purpose=inline)
- [keylabs.ai](https://keylabs.ai/blog/revolutionizing-autonomous-driving-with-instance-segmentation/?utm_source=chatgpt.com)

Instead of assigning the same label to every car, we assign a unique ID to each individual object.

Example:

```python
instance_ids = {
    0: "Background",
    1: "Car 1",
    2: "Car 2",
    3: "Car 3"
}
```

Simplified instance mask:

```text
0 0 0 0 0 0
0 1 1 0 2 2
0 1 1 0 2 2
0 0 0 0 0 0
0 3 3 3 0 0
0 3 3 3 0 0
```

Here:

- `0` → Background
- `1` → First car
- `2` → Second car
- `3` → Third car

Mathematically:

$$
M \in \{0,1,2,\ldots,K\}^{H \times W}
$$

Where \\(K\\) is the number of object instances.

**Important distinction:** Instance IDs identify individual objects, but do not necessarily tell us their semantic class. A separate mapping can associate each instance ID with its class.

For example:

| Instance ID | Semantic class |
|---|---|
| 1 | Car |
| 2 | Car |
| 3 | Person |
| 4 | Bicycle |

### 4.4 Compare All Three Mask Representations

![Visual comparison of binary, semantic and instance segmentation masks](https://bioimagingai.janelia.org/9-figures/chapter9_1_segmentation_types.png)

*Source: [Janelia — AI in Microscopy](https://bioimagingai.janelia.org/9-train-models.html)*

**Original reference:**

- [Joy Hsu](https://images.openai.com/static-rsc-4/I27gvbgviNbg9FRLw7uhpMjyCGpYLkC90h1dw-MRaYaoMKldB-ZrLM37IUcI-e72Ilde7nnv8GwAUKMqEBuIi54Wcf7okXsw8HYfl-T0SfqQ8sBg29t1gzjEJgRvciK1c3a1A_HRRsm2-4GIIOj8vYmusrO61sNehjCXQknlIlo?purpose=inline)
- [web.stanford.edu](https://web.stanford.edu/~joycj/?utm_source=chatgpt.com)

| Feature | Binary | Multi-class | Instance IDs |
|---|---|---|---|
| Values | 0, 1 | 0 to C−1 | 0 to K |
| Purpose | Foreground/background | Identify classes | Identify individual objects |
| Multiple classes | No | Yes | Not inherently |
| Separate object identity | No | No | Yes |
| Typical task | Binary segmentation | Semantic segmentation | Instance segmentation |

### 4.5 Practical PyTorch Implementation

Let's create these mask representations using NumPy and PyTorch.

```python
import numpy as np
import torch

# 1. Binary mask
binary_mask = np.array([
    [0, 0, 0, 0],
    [0, 1, 1, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0]
])

# 2. Multi-class mask
semantic_mask = np.array([
    [0, 0, 1, 1],
    [0, 2, 2, 1],
    [0, 3, 3, 1],
    [0, 0, 0, 0]
])

# 3. Instance ID mask
instance_mask = np.array([
    [0, 0, 0, 0],
    [0, 1, 1, 2],
    [0, 1, 1, 2],
    [0, 0, 3, 3]
])

# Convert into PyTorch tensors
binary_mask = torch.tensor(
    binary_mask,
    dtype=torch.float32
)

semantic_mask = torch.tensor(
    semantic_mask,
    dtype=torch.long
)

instance_mask = torch.tensor(
    instance_mask,
    dtype=torch.long
)

print("Binary:", binary_mask.shape)
```

Expected output:

```text
Binary: torch.Size([4, 4])
Semantic: torch.Size([4, 4])
Instance: torch.Size([4, 4])
```

Notice that all three masks have identical spatial dimensions. What changes is the meaning of their pixel values.

### 4.6 Important Industry Concept: Semantic Labels vs Instance Labels

In real-world instance segmentation datasets, we often maintain two kinds of information:

1. **Semantic class:** What category does the object belong to?
2. **Instance ID:** Which individual object is it?

For example:

| Pixel | Semantic class | Instance ID |
|---|---|---|
| Pixel A | Car | 1 |
| Pixel B | Car | 1 |
| Pixel C | Car | 2 |
| Pixel D | Person | 3 |

This distinction becomes particularly useful when working with datasets such as COCO and implementing Mask R-CNN.

## What You Should Remember 🧠

1. Binary masks represent foreground and background using 0 and 1.
2. Multi-class masks assign a class ID to every pixel.
3. Instance masks assign individual IDs to separate objects.
4. Semantic class IDs and instance IDs represent different information.
5. Segmentation masks are numerical label maps, not ordinary RGB images.
6. PyTorch commonly uses `torch.long` target masks for multi-class Cross-Entropy Loss.

---
